
# TP RAG en entreprise : assistant documentaire Altéa Services By NDISSA MAPELA
Objectif : un assistant qui répond aux questions des salariés à partir de la documentation interne, en citant la source.


# Exercice 1  Préparation des documents

## Étape 1  Collecte

On lit chaque fichier et on garde pour chaque page son texte, son fichier, son numéro de page et son service.

### Chemins du projet


In [ ]:
from pathlib import Path

RACINE = Path("..")                              # le notebook est dans notebook/, la racine est au-dessus
DOSSIER_DATA = RACINE / "data"                   # documents (General, IT, RH)
DOSSIER_BASE = RACINE / "base_vectorielle"       # base ChromaDB
FICHIER_EVAL = RACINE / "evaluation.csv"         # jeu d'évaluation


## Collecte des documents 

In [6]:
from pathlib import Path
from pypdf import PdfReader

def lire_pdf(chemin):
    lecteur = PdfReader(chemin)
    pages = []
    for numero, page in enumerate(lecteur.pages, start=1):
        texte = page.extract_text()                       # texte brut de la page
        pages.append({"source": chemin.name, "page": numero, "texte": texte})
    return pages

def lire_txt(chemin):
    texte = chemin.read_text(encoding="utf-8")
    return [{"source": chemin.name, "page": 1, "texte": texte}]   # un .txt = une seule "page"

def collecter(dossier=DOSSIER_DATA):
    pages = []
    for chemin in sorted(Path(dossier).rglob("*")):
        service = chemin.parent.name                      # RH, IT ou General
        if chemin.suffix == ".pdf":
            pages_fichier = lire_pdf(chemin)
        elif chemin.suffix == ".txt":
            pages_fichier = lire_txt(chemin)
        else:
            continue                                      # on ignore les dossiers et autres fichiers
        for p in pages_fichier:
            p["service"] = service
            pages.append(p)
    return pages

pages = collecter()
print(len(pages), "pages lues")

18 pages lues


**Q1.** Nombre de pages lues par document :

In [7]:

from collections import Counter

compte = Counter((p["service"], p["source"]) for p in pages)
for (service, source), nb in sorted(compte.items()):
    print(f"{service:<8} {source:<25} {nb} page(s)")

General  horaires.txt              1 page(s)
General  plan_acces.txt            1 page(s)
General  reglement_interieur.pdf   2 page(s)
General  restaurant.txt            1 page(s)
General  securite_incendie.pdf     1 page(s)
IT       demande_materiel.txt      1 page(s)
IT       imprimantes.txt           1 page(s)
IT       mot_de_passe.txt          1 page(s)
IT       securite_emails.pdf       1 page(s)
IT       vpn.pdf                   1 page(s)
RH       conges.pdf                2 page(s)
RH       formation.txt             1 page(s)
RH       mutuelle.txt              1 page(s)
RH       notes_de_frais.pdf        1 page(s)
RH       teletravail.pdf           2 page(s)


**Pourquoi garder le fichier, la page et le service ?**

Parce que le texte seul ne permet pas de savoir d'où vient une information, 
On a besoin de la source notamment de la page, du service et du fichier dans le but que l'agent précise dans quel document et à quelle page il a trouvé la réponse.



## Étape 2   Nettoyage

In [ ]:
import re

def nettoyer(texte):
    texte = re.sub(r"[ \t]+", " ", texte)            # plusieurs espaces -> un seul
    texte = re.sub(r"\n{3,}", "\n\n", texte)         # 3 sauts de ligne ou plus -> 2
    texte = re.sub(r"-\n(\w)", r"\1", texte)          # "dévelop-\npement" -> "développement"
    return texte.strip()

In [ ]:
**Q3.** On nettoie toutes les pages et on supprime les pages devenues vides.

In [9]:
pages_brutes = [dict(p) for p in pages]              # copie, pour comparer avant/après (Q4)

for p in pages:
    p["texte"] = nettoyer(p["texte"])
pages = [p for p in pages if p["texte"]]             # on garde seulement les pages non vides

print(len(pages), "pages après nettoyage")

18 pages après nettoyage


## Étape 3  Découpage (chunking)

Un document entier est trop long pour le LLM : on le coupe en morceaux (chunks) qui se chevauchent un peu, pour ne pas perdre une phrase coupée en deux.

In [ ]:
**Q4.** Une même page avant et après nettoyage :

In [ ]:
i = next(n for n, p in enumerate(pages_brutes) if p["source"] == "teletravail.pdf" and p["page"] == 2)

print(" AVANT ")
print(repr(pages_brutes[i]["texte"]))
print("\n APRÈS ")
print(repr(pages[i]["texte"]))

 AVANT 
'Altéa Services - Document interne\nIndemnité et équipement\nUne indemnité de 2,50 euros par jour télétravaillé est versée avec le salaire, dans\nla limite de 50 euros par mois. Altéa fournit un ordinateur portable et un casque. La\nconnexion aux outils internes se fait obligatoirement par le VPN.\nPage 2 / 2\n'

 APRÈS 
'Altéa Services - Document interne\nIndemnité et équipement\nUne indemnité de 2,50 euros par jour télétravaillé est versée avec le salaire, dans\nla limite de 50 euros par mois. Altéa fournit un ordinateur portable et un casque. La\nconnexion aux outils internes se fait obligatoirement par le VPN.\nPage 2 / 2'


**Q4 Défaut restant et règle de correction**

L'**en-tête répété** (« Altéa Services - Document interne ») et le **numéro de page** (« Page 2 / 2 ») sont toujours là. Ils se retrouveront dans les chunks et ajoutent du bruit.

Règle proposée : supprimer les lignes qui se répètent sur toutes les pages, et les lignes de la forme « Page X / Y » :
```python
texte = re.sub(r"^Altéa Services - Document interne\n", "", texte)
texte = re.sub(r"\nPage \d+ / \d+$", "", texte)
```

## Étape 3 — Découpage (chunking)

Un document entier est trop long pour le LLM : on le coupe en morceaux (chunks) qui se chevauchent un peu, pour ne pas perdre une phrase coupée en deux.

In [14]:
def decouper(texte, taille=500, chevauchement=50):
    assert chevauchement < taille, "le chevauchement doit être plus petit que la taille"
    morceaux = []
    debut = 0
    while debut < len(texte):
        fin = debut + taille                          # fin du morceau
        morceaux.append(texte[debut:fin])
        if fin >= len(texte):
            break
        debut = fin - chevauchement                   # on recule pour garder le chevauchement
    return morceaux

def decouper_pages(pages, taille=500, chevauchement=50):
    chunks = []
    for p in pages:
        for i, morceau in enumerate(decouper(p["texte"], taille, chevauchement)):
            chunks.append({
                "id": f"{p['source']}-p{p['page']}-{i}",
                "texte": morceau,
                "source": p["source"],
                "page": p["page"],
                "service": p["service"],
            })
    return chunks

**Q5.** Vérification :

In [16]:
print(decouper("abcdefghij", taille=4, chevauchement=1))
assert decouper("abcdefghij", taille=4, chevauchement=1) == ["abcd", "defg", "ghij"]

['abcd', 'defg', 'ghij']


**Q5  Explication à la main** (taille 4, chevauchement 1, donc on avance de 3 à chaque fois) :

| Tour | début | fin | Morceau | Suite |
|---|---|---|---|---|
| 1 | 0 | 4 | `abcd` | début = 4 − 1 = 3 |
| 2 | 3 | 7 | `defg` | début = 7 − 1 = 6 |
| 3 | 6 | 10 | `ghij` | fin = 10 = longueur du texte → arrêt |

Chaque morceau reprend la dernière lettre du précédent (`d`, puis `g`) : c'est le chevauchement.

**Q6.** Découpage du corpus avec trois tailles :

In [21]:
print(f"{'Taille':>7} {'Chevauchement':>14} {'Nb chunks':>10} {'Long. moyenne':>14}")
for taille, chev in [(200, 20), (500, 50), (1000, 100)]:
    ch = decouper_pages(pages, taille, chev)
    moyenne = sum(len(c["texte"]) for c in ch) / len(ch)
    print(f"{taille:>7} {chev:>14} {len(ch):>10} {moyenne:>14.0f}")

 Taille  Chevauchement  Nb chunks  Long. moyenne
    200             20         38            166
    500             50         18            329
   1000            100         18            329


**Q7**

Chunk trop petit : on perd le contexte et on ne comprend pas bien l'information.

Chunk trop grand : il mélange plusieurs informations et rend la recherche moins précise.

Chevauchement : il permet de ne pas perdre des informations entre deux morceaux de texte.

Taille retenue : 500 caractères avec 50 caractères de chevauchement. Cela permet de garder suffisamment de contexte tout en facilitant la recherche.

In [20]:
TAILLE, CHEVAUCHEMENT = 500, 50
chunks = decouper_pages(pages, TAILLE, CHEVAUCHEMENT)
print(len(chunks), "chunks retenus")

18 chunks retenus


## Étape 4  Embeddings et indexation

### Partie A  Découvrir les embeddings

Un modèle d'embedding transforme un texte en vecteur c-a-d une liste de nombres. Deux textes de sens proche ont des vecteurs proches.

In [24]:
from sentence_transformers import SentenceTransformer, util

modele = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

phrases = [
    "Comment poser un jour de congé ?",
    "Quelle est la procédure pour demander des vacances ?",
    "Mon ordinateur ne se connecte pas au VPN.",
]
vecteurs = modele.encode(phrases)
print(vecteurs.shape)
print(util.cos_sim(vecteurs, vecteurs))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1315.49it/s]


(3, 384)
tensor([[ 1.0000,  0.4888, -0.0353],
        [ 0.4888,  1.0000, -0.0020],
        [-0.0353, -0.0020,  1.0000]])


**Q8.** Dimension d'un vecteur et nombre de valeurs à stocker :

In [25]:
dimension = vecteurs.shape[1]
print("Dimension d'un vecteur :", dimension)
print("Nombre de chunks :", len(chunks))
print("Nombres à stocker :", len(chunks), "x", dimension, "=", len(chunks) * dimension)

Dimension d'un vecteur : 384
Nombre de chunks : 18
Nombres à stocker : 18 x 384 = 6912


**Q9  Lecture de la matrice de similarité**

La diagonale vaut 1, car chaque phrase est comparée à elle-même.



Les phrases 1 et 2 se ressemblent le plus, car elles parlent de la même chose.



La phrase 3 (VPN) est différente des deux autres.

### Partie B  Ranger les chunks dans la base vectorielle

**Q10.** Les étapes 1 à 4 sont enchaînées dans ce notebook : `chunks` vient de l'étape 3. On les vectorise et on les range dans ChromaDB.

In [26]:
import chromadb

client = chromadb.PersistentClient(path=str(DOSSIER_BASE))
collection = client.get_or_create_collection("altea", metadata={"hnsw:space": "cosine"})

textes = [c["texte"] for c in chunks]
vecteurs = modele.encode(textes, show_progress_bar=True).tolist()

collection.upsert(
    ids=[c["id"] for c in chunks],
    documents=textes,
    embeddings=vecteurs,
    metadatas=[{"source": c["source"], "page": c["page"], "service": c["service"]} for c in chunks],
)
print(collection.count(), "chunks indexés")

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.31it/s]

18 chunks indexés


In [27]:
# Vérification Q10 : autant de chunks indexés que de chunks produits
assert collection.count() == len(chunks)
print("OK :", collection.count(), "=", len(chunks))

OK : 18 = 18


**Q11.** Première recherche : les 3 chunks les plus proches pour 3 questions.

In [28]:
questions = [
    "Combien de jours de télétravail ?",
    "Comment réinitialiser mon mot de passe ?",
    "Où se trouve le point de rassemblement en cas d'incendie ?",
]
for q in questions:
    resultat = collection.query(query_embeddings=modele.encode([q]).tolist(), n_results=3)
    print("\nQUESTION :", q)
    for doc, meta, dist in zip(resultat["documents"][0], resultat["metadatas"][0], resultat["distances"][0]):
        print(f"  - {meta['source']} p.{meta['page']} (distance {dist:.3f}) : {doc[:80]}...")


QUESTION : Combien de jours de télétravail ?
  - teletravail.pdf p.1 (distance 0.323) : Altéa Services - Document interne
Charte de télétravail
Le télétravail est ouver...
  - conges.pdf p.1 (distance 0.541) : Altéa Services - Document interne
Congés et absences
Chaque salarié acquiert 2,5...
  - horaires.txt p.1 (distance 0.570) : Horaires de travail

Les locaux sont ouverts de 7 h 30 à 19 h 30, du lundi au ve...

QUESTION : Comment réinitialiser mon mot de passe ?
  - mot_de_passe.txt p.1 (distance 0.268) : Mot de passe oublié

Pour réinitialiser votre mot de passe, allez sur le portail...
  - securite_emails.pdf p.1 (distance 0.767) : Altéa Services - Document interne
Sécurité des e-mails
Ne cliquez jamais sur un ...
  - demande_materiel.txt p.1 (distance 0.816) : Demande de matériel

Les demandes de matériel informatique (écran, clavier, sour...

QUESTION : Où se trouve le point de rassemblement en cas d'incendie ?
  - securite_incendie.pdf p.1 (distance 0.472) : Altéa Services - 

**Q12 Relancer l'indexation**

Relancer l'indexation ne double pas le nombre de chunks grâce à upsert.



upsert permet de mettre à jour un chunk s'il existe déjà ou de l'ajouter s'il n'existe pas.



Avec add, on risque de créer des doublons.

---
# Exercice 2  Interroger, évaluer et livrer

## Étape 5  Recherche

On transforme la question en vecteur, puis on demande à la base les chunks les plus proches.

In [29]:
collection = chromadb.PersistentClient(path=str(DOSSIER_BASE)).get_collection("altea")

def rechercher(question, k=3, service=None):
    vecteur = modele.encode([question]).tolist()
    filtre = {"service": service} if service else None
    res = collection.query(query_embeddings=vecteur, n_results=k, where=filtre)
    resultats = []
    for texte, meta, distance in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        resultats.append({
            "texte": texte,
            "source": meta["source"],
            "page": meta["page"],
            "service": meta["service"],
            "score": round(1 - distance, 3),     # plus le score est proche de 1, plus c'est proche
        })
    return resultats

def afficher(resultats):
    for r in resultats:
        print(f"  - {r['source']} p.{r['page']} [{r['service']}] score={r['score']}")

**Q1.** Résultats pour 3 questions de salariés :

In [30]:
for q in ["Quel est le plafond pour un repas en note de frais ?",
          "Comment me connecter au VPN depuis chez moi ?",
          "À quelle heure ouvre le restaurant d'entreprise ?"]:
    print("\nQUESTION :", q)
    afficher(rechercher(q))


QUESTION : Quel est le plafond pour un repas en note de frais ?
  - restaurant.txt p.1 [General] score=0.469
  - horaires.txt p.1 [General] score=0.326
  - notes_de_frais.pdf p.1 [RH] score=0.233

QUESTION : Comment me connecter au VPN depuis chez moi ?
  - vpn.pdf p.1 [IT] score=0.714
  - teletravail.pdf p.2 [RH] score=0.371
  - mutuelle.txt p.1 [RH] score=0.213

QUESTION : À quelle heure ouvre le restaurant d'entreprise ?
  - restaurant.txt p.1 [General] score=0.73
  - horaires.txt p.1 [General] score=0.635
  - teletravail.pdf p.1 [RH] score=0.454


**Q2.** Même question avec k = 1, 3 et 10 :

In [31]:
q = "Combien de jours de télétravail par semaine ?"
for k in (1, 3, 10):
    print(f"\nk = {k}")
    afficher(rechercher(q, k=k))


k = 1
  - teletravail.pdf p.1 [RH] score=0.644

k = 3
  - teletravail.pdf p.1 [RH] score=0.644
  - conges.pdf p.1 [RH] score=0.482
  - horaires.txt p.1 [General] score=0.442

k = 10
  - teletravail.pdf p.1 [RH] score=0.644
  - conges.pdf p.1 [RH] score=0.482
  - horaires.txt p.1 [General] score=0.442
  - notes_de_frais.pdf p.1 [RH] score=0.384
  - conges.pdf p.2 [RH] score=0.364
  - formation.txt p.1 [RH] score=0.363
  - teletravail.pdf p.2 [RH] score=0.345
  - demande_materiel.txt p.1 [IT] score=0.336
  - restaurant.txt p.1 [General] score=0.312
  - plan_acces.txt p.1 [General] score=0.273


Q2  Impact de la valeur de k
- Quand k augmente, on récupère plus de passages : on a plus de chances de trouver la bonne réponse.
- Mais on risque aussi de récupérer des passages inutiles ou hors sujet.
- Cela rend la réponse plus longue, plus lente et parfois moins précise.
Conclusion : k = 3 est un bon compromis entre quantité et précision des informations.

**Q3.** Une question RH, filtrée sur RH puis sur IT :

In [33]:
q = "Combien de jours de congé pour un mariage ?"
print("service = RH")
afficher(rechercher(q, service="RH"))
print("\nservice = IT")
afficher(rechercher(q, service="IT"))

service = RH
  - conges.pdf p.1 [RH] score=0.5
  - conges.pdf p.2 [RH] score=0.446
  - formation.txt p.1 [RH] score=0.388

service = IT
  - demande_materiel.txt p.1 [IT] score=0.286
  - imprimantes.txt p.1 [IT] score=0.239
  - mot_de_passe.txt p.1 [IT] score=0.196


Q3 – Filtrer les documents par service
- Avec service="RH", la recherche trouve uniquement les documents RH, dont conges.pdf qui contient la réponse.
- Avec service="IT", la recherche trouve uniquement les documents IT. Le bon document n'est donc pas trouvé.

Concernant la confidentialité :
-Le filtre permet de limiter l'accès aux documents selon le service.

Mais ce n'est pas suffisant : il faut aussi vérifier l'identité de l'utilisateur et ses droits d'accès pour être sûr qu'il ne consulte que les documents autorisés.

## Étape 6  Génération

On donne les chunks trouvés au LLM pour qu'il rédige une réponse sourcée. Le LLM tourne en local avec **Ollama** 

In [34]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(RACINE / ".env")                    

client_llm = OpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),       # adresse du LLM (Ollama en local)
    api_key=os.getenv("OPENAI_API_KEY"),         # clé lue dans .env, jamais écrite dans le code
)
MODELE = os.getenv("MODELE")
REPLI = "Je ne trouve pas cette information dans la documentation."

def construire_prompt(question, extraits):
    contexte = ""
    for i, e in enumerate(extraits, start=1):
        contexte += f"[{i}] ({e['source']}, page {e['page']})\n{e['texte']}\n\n"
    return f"""Tu es l'assistant interne d'Altéa Services.
Réponds à la question en utilisant UNIQUEMENT les extraits ci-dessous.
Après chaque information, indique le numéro de l'extrait entre crochets, par exemple [1].
Si les extraits ne contiennent pas la réponse, réponds exactement : "{REPLI}"

Extraits :
{contexte}
Question : {question}
Réponse :"""

def repondre(question, k=3):
    extraits = rechercher(question, k)
    prompt = construire_prompt(question, extraits)
    reponse = client_llm.chat.completions.create(
        model=MODELE,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    return reponse.choices[0].message.content, extraits

**Q4.** Le prompt complet envoyé pour une question :

In [35]:
q = "Combien de jours de télétravail par semaine ?"
print(construire_prompt(q, rechercher(q)))

Tu es l'assistant interne d'Altéa Services.
Réponds à la question en utilisant UNIQUEMENT les extraits ci-dessous.
Après chaque information, indique le numéro de l'extrait entre crochets, par exemple [1].
Si les extraits ne contiennent pas la réponse, réponds exactement : "Je ne trouve pas cette information dans la documentation."

Extraits :
[1] (teletravail.pdf, page 1)
Altéa Services - Document interne
Charte de télétravail
Le télétravail est ouvert aux salariés ayant terminé leur période d'essai. Il est
autorisé jusqu'à 2 jours par semaine, fixés avec le manager. Le jeudi est un jour de
présence obligatoire pour toutes les équipes.
Les jours de télétravail sont déclarés dans MonAltéa, menu Télétravail.
Page 1 / 2

[2] (conges.pdf, page 1)
Altéa Services - Document interne
Congés et absences
Chaque salarié acquiert 2,5 jours ouvrables de congés payés par mois, soit 30 jours
par an. Les congés se posent dans l'outil RH MonAltéa, menu Absences, au moins 15
jours avant le départ.
Le ma

Les trois parties du prompt :
1. **Consigne** : de « Tu es l'assistant… » jusqu'à la phrase de repli ;
2. **Extraits** : les chunks numérotés `[1]`, `[2]`, `[3]` avec leur source et leur page ;
3. **Question** : la question du salarié, suivie de « Réponse : ».

**Q5.** 5 questions, dont 2 sans réponse dans la documentation :

In [37]:
questions_q5 = [
    "Combien de jours de télétravail par semaine ?",
    "Quel est le plafond pour une nuit d'hôtel à Paris ?",
    "Comment réinitialiser mon mot de passe ?",
    "Quel temps fera-t-il demain ?",                       # sans réponse
    "Quel est le salaire du directeur ?",                  # sans réponse
]
for q in questions_q5:
    reponse, extraits = repondre(q)
    print("QUESTION :", q)
    print("RÉPONSE  :", reponse)
    print("SOURCES  :", [f"{e['source']} p.{e['page']}" for e in extraits])
    print("-" * 80)

QUESTION : Combien de jours de télétravail par semaine ?
RÉPONSE  : Selon la charte de télétravail, le télétravail est autorisé jusqu'à 2 jours par semaine, fixés avec le manager.
SOURCES  : ['teletravail.pdf p.1', 'conges.pdf p.1', 'horaires.txt p.1']
--------------------------------------------------------------------------------
QUESTION : Quel est le plafond pour une nuit d'hôtel à Paris ?
RÉPONSE  : Le plafond pour une nuit d'hôtel à Paris est de 110 euros. [3]
SOURCES  : ['restaurant.txt p.1', 'plan_acces.txt p.1', 'notes_de_frais.pdf p.1']
--------------------------------------------------------------------------------
QUESTION : Comment réinitialiser mon mot de passe ?
RÉPONSE  : Pour réinitialiser votre mot de passe, allez sur le portail reset.altea.example et cliquez sur « Mot de passe oublié ». Un code de validation est envoyé sur votre téléphone professionnel.
SOURCES  : ['mot_de_passe.txt p.1', 'securite_emails.pdf p.1', 'demande_materiel.txt p.1']
------------------------

**Q6.** La même question posée directement au LLM, sans extraits :

In [38]:
q = "Combien de jours de télétravail par semaine sont autorisés chez Altéa ?"

sans_rag = client_llm.chat.completions.create(
    model=MODELE, temperature=0,
    messages=[{"role": "user", "content": q}],
).choices[0].message.content

avec_rag, _ = repondre(q)

print("SANS RAG :", sans_rag)
print("\nAVEC RAG :", avec_rag)

SANS RAG : Je ne peux pas fournir d'informations sur les politiques d'entreprise de Altéa.

AVEC RAG : 2 jours par semaine, fixés avec le manager.


Q6 – Pourquoi utiliser le RAG ?
Sans RAG, le LLM ne connaît pas les documents internes et risque d'inventer des réponses.

Avec le RAG, il trouve la bonne information dans les documents et cite sa source.

Une hallucination, c'est une information fausse ou inventée présentée comme vraie.

En résumé : le RAG permet d'avoir des réponses plus fiables, basées sur les documents et vérifiables grâce aux sources.
Q7 – Pourquoi utiliser temperature=0 ?
temperature=0 permet d'obtenir des réponses plus stables et prévisibles.

Le LLM évite de laisser trop de place à la créativité et limite ainsi les risques d'inventer des informations.

En résumé : on privilégie des réponses précises et fidèles aux documents plutôt que créatives.

## Étape 7  Évaluation

On mesure d'abord la recherche (le bon document est-il trouvé ?), puis les réponses.

**Q8.** Création de `evaluation.csv` (à la racine du projet) : 10 questions et le document qui contient la réponse.

In [39]:
import csv

evaluation = [
    ("Combien de jours de télétravail par semaine ?", "teletravail.pdf"),
    ("Comment réinitialiser mon mot de passe ?", "mot_de_passe.txt"),
    ("Combien de jours de congé pour un mariage ?", "conges.pdf"),
    ("Quel est le plafond pour un repas en note de frais ?", "notes_de_frais.pdf"),
    ("Quelle part de la mutuelle est payée par Altéa ?", "mutuelle.txt"),
    ("Comment me connecter aux applications depuis chez moi ?", "vpn.pdf"),
    ("Comment demander un nouvel écran ?", "demande_materiel.txt"),
    ("Que faire si je reçois un e-mail suspect ?", "securite_emails.pdf"),
    ("À quelle heure ouvrent les locaux ?", "horaires.txt"),
    ("Où se trouve le point de rassemblement en cas d'alarme ?", "securite_incendie.pdf"),
]

with open(FICHIER_EVAL, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["question", "document_attendu"])
    writer.writerows(evaluation)

print(FICHIER_EVAL.read_text(encoding="utf-8"))

question,document_attendu
Combien de jours de télétravail par semaine ?,teletravail.pdf
Comment réinitialiser mon mot de passe ?,mot_de_passe.txt
Combien de jours de congé pour un mariage ?,conges.pdf
Quel est le plafond pour un repas en note de frais ?,notes_de_frais.pdf
Quelle part de la mutuelle est payée par Altéa ?,mutuelle.txt
Comment me connecter aux applications depuis chez moi ?,vpn.pdf
Comment demander un nouvel écran ?,demande_materiel.txt
Que faire si je reçois un e-mail suspect ?,securite_emails.pdf
À quelle heure ouvrent les locaux ?,horaires.txt
Où se trouve le point de rassemblement en cas d'alarme ?,securite_incendie.pdf



**Q9.** Taux de réussite de la recherche : le bon document est-il parmi les k résultats ?

In [40]:
def taux_reussite(fichier=FICHIER_EVAL, k=3):
    with open(fichier, encoding="utf-8") as f:
        lignes = list(csv.DictReader(f))
    trouves = 0
    for ligne in lignes:
        sources = [r["source"] for r in rechercher(ligne["question"], k)]
        if ligne["document_attendu"] in sources:          # le document attendu est-il trouvé ?
            trouves += 1
    return trouves / len(lignes)

for k in (1, 3, 5):
    print(f"k = {k} : {taux_reussite(k=k):.0%} de questions où le bon document est trouvé")

k = 1 : 80% de questions où le bon document est trouvé
k = 3 : 100% de questions où le bon document est trouvé
k = 5 : 100% de questions où le bon document est trouvé


**Q10.** Les 10 réponses générées, à noter :

In [41]:
with open(FICHIER_EVAL, encoding="utf-8") as f:
    lignes = list(csv.DictReader(f))

for n, ligne in enumerate(lignes, start=1):
    reponse, extraits = repondre(ligne["question"])
    sources = [e["source"] for e in extraits]
    print(f"{n}. {ligne['question']}")
    print("   Réponse :", reponse)
    print("   Attendu :", ligne["document_attendu"], "| trouvé :", ligne["document_attendu"] in sources)
    print()

1. Combien de jours de télétravail par semaine ?
   Réponse : 2 jours par semaine, fixés avec le manager.
   Attendu : teletravail.pdf | trouvé : True

2. Comment réinitialiser mon mot de passe ?
   Réponse : Pour réinitialiser votre mot de passe, allez sur le portail reset.altea.example et cliquez sur « Mot de passe oublié ». Un code de validation est envoyé sur votre téléphone professionnel.
   Attendu : mot_de_passe.txt | trouvé : True

3. Combien de jours de congé pour un mariage ?
   Réponse : Selon l'extrait [1], pour un mariage, un salarié bénéficie de 4 jours de congés exceptionnels.
   Attendu : conges.pdf | trouvé : True

4. Quel est le plafond pour un repas en note de frais ?
   Réponse : Selon les notes de frais, le plafond pour un repas en note de frais est de 20 euros. [3]
   Attendu : notes_de_frais.pdf | trouvé : True

5. Quelle part de la mutuelle est payée par Altéa ?
   Réponse : Altéa prend en charge 60 % de la cotisation de la mutuelle SantéPlus. [2]
   Attendu : m

In [42]:
q = "Comment me connecter aux applications depuis chez moi ?"
for k in (3, 5):
    print(f"k = {k} :", [r["source"] for r in rechercher(q, k)])

print(f"\nTaux de réussite : k=3 -> {taux_reussite(k=3):.0%} | k=5 -> {taux_reussite(k=5):.0%}")

k = 3 : ['vpn.pdf', 'plan_acces.txt', 'teletravail.pdf']
k = 5 : ['vpn.pdf', 'plan_acces.txt', 'teletravail.pdf', 'teletravail.pdf', 'demande_materiel.txt']

Taux de réussite : k=3 -> 100% | k=5 -> 100%


## Étape 8 — Mise à disposition et GitHub

**Q12.** On crée `app.py` **à la racine du projet** Il reprend les fonctions `rechercher` et `repondre` pour être autonome, et se lance depuis la racine : ses chemins partent donc de la racine.

Lancement, dans le terminal, **depuis la racine du projet** :
```
streamlit run app.py
```
La page s'ouvre dans le navigateur. *Insérer ici la capture d'écran d'une réponse avec ses sources.*